# Tools
- A tool is just a Python function (or API) that is packaged in a way the LLM can understand and call when needed.
- LLMs like GPT are great at reasoning & generation, but they can't do things like
    1. Access live data
    2. Do reliable math
    3. Calls APIs
    4. Run Code
    5. Interact with a database

- Type of Tools:
    1. Built-in Tools
    2. Custom Tools

- An AI agent is an LLM-powered system that can autonomously think, decide, and take actions using external tools or APIs to achieve a goal

### Built-in Tools:
- A built-in tool is a tool that LangChain already provides for you —it’s pre-built, production ready, and requires minimal or no setup.
- You don’t have to write the function logic yourself — you just import and use it.
- Some Tools:
    1. DuckDuckGoSearchRun -> Web search via DuckDuckGo
    2. WikipediaQueryRun -> Wikipedia Summary
    3. PythonREPLTool -> Run raw Python code
    4. ShellTool -> Run shell commands
    5. GmailSendMessageTool -> Send emails via Gmail
    6. SlackSendMessage -> Post message to slack
    7. SQLDatabaseQueryTool -> Run SQL Queries

In [2]:
# 1. DuckDuckGoSearchRun Tool --> This tool allows you to perform a search using the DuckDuckGo search engine. You can use it to retrieve information on various topics, such as news, weather, or general knowledge. The tool takes a query as input and returns the search results.
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke('top news in india today')

print("----Tool Name------\n",search_tool.name)
print("----Tool Description-----\n", search_tool.description)
print("----Tool Arguments-----------\n", search_tool.args)
print("----Search Results-----\n", results)


----Tool Name------
 duckduckgo_search
----Tool Description-----
 A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
----Tool Arguments-----------
 {'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}
----Search Results-----
 5 hours ago - Behind month of labour shortage ... ... On the anniversary of the Pahalgam attack, PM Modi reaffirmed India's resolve, stating the nation will never bow to terror and honoring the 26 lives lost in the "heinous" act... 11 hours ago - India News: Read Latest India News Today India Top Headlines along with Latest Breaking India News and Real Time announcements from India. Stay connected for timely reports and detailed India News coverage every day. 1 hour ago - Anne Hathaway recently spoke about ageing and life in an interview, where her use of the phrase "Inshallah" caught attention online. She will be seen next in The Devil Wears Pr

In [5]:
# 2. ShellTool --> This tool allows you to execute shell commands directly from your code. It can be used to perform various tasks, such as listing files in a directory, checking system information, or running scripts. The tool takes a shell command as input and returns the output of that command.
from langchain_community.tools import ShellTool

shell_tool = ShellTool()
results = shell_tool.invoke('cd')

print("----Shell Tool Results---\n", results)

Executing command:
 cd
----Shell Tool Results---
 d:\Repos\Gen-AI\langchain



### Custom Tools:
- A custom tool is a tool that you define yourself.
- We use them when:
    1. You want to call your own APIs
    2. You want to encapsulate business logic
    3. You want the LLM to interact with your database, product, or app

### Ways to create Custom tool:
1. Using @tool decorator:
    - It  is a special type of tool where the input to the tool follows a structured schema, typically defined using a Pydantic model
2. Using StructuredTool & Pydantic:
    - All other tool types like @tool, StructuredTool are built on top of BaseTool
3. Using BaseTool Class:
    - It is the abstract base class for all tools in LangChain. It defines the core structure and interface that any tool must follow, whether it's a simple one-liner or a fully customized function.

### @tool decorator
Creating a tool using a tool decorator

In [ ]:
"""
Step 1 - create a function
Step 2 - add type hints
Step 3 - add tool decorator
"""

from langchain_core.tools import tool

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

result = multiply.invoke({"a":3, "b":5})

print("---Tool Name---\n", multiply.name)
print("---Tool Description---\n", multiply.description)
print("---Tool Arguments---\n", multiply.args)
print("---Result---\n", result)

---Tool Name---
 multiply
---Tool Description---
 Multiply two numbers
---Tool Arguments---
 {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
---Result---
 15


### StructuredTool
creating a tool Using StructuredTool

In [7]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

def multiply_func(a: int, b: int) -> int:
    return a * b

multiply = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

result = multiply.invoke({"a":3, "b":3})

print("---Tool Name---\n", multiply.name)
print("---Tool Description---\n", multiply.description)
print("---Tool Arguments---\n", multiply.args)
print("---Result---\n", result)

---Tool Name---
 multiply
---Tool Description---
 Multiply two numbers
---Tool Arguments---
 {'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}
---Result---
 9


C:\Users\divya\AppData\Local\Temp\ipykernel_29064\2365871837.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\divya\AppData\Local\Temp\ipykernel_29064\2365871837.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


###  BaseTool Class
Creating a tool  Using BaseTool Class

In [8]:
from langchain.tools import BaseTool
from typing import Type

# arg schema using pydantic
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"
    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b
    
multiply = MultiplyTool()
result = multiply.invoke({'a':3, 'b':3})

print("---Tool Name---\n", multiply.name)
print("---Tool Description---\n", multiply.description)
print("---Tool Arguments---\n", multiply.args)
print("---Result---\n", result)

d:\Repos\Gen-AI\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---Tool Name---
 multiply
---Tool Description---
 Multiply two numbers
---Tool Arguments---
 {'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}
---Result---
 9


C:\Users\divya\AppData\Local\Temp\ipykernel_29064\498108010.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\divya\AppData\Local\Temp\ipykernel_29064\498108010.py:7: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


## Toolkits:
- A toolkit is just a collection (bundle) of related tools that serve a common purpose packaged together for convenience and reusability.
- In LangChain A toolkit might be: GoogleDriveToolKit And it can contain the following tools:
    1. GoogleDriveCreateFileTool - Upload a file
    2. GoogleDriveSearchTool - Search for a file by name/content
    3. GoogleDriveReadFile - Read contents of a file

In [9]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def sub(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

class MathToolkit:
    def get_tools(self):
        return [add, sub, multiply]

toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
sub => Subtract two numbers
multiply => Multiply two numbers
